# C001: Full Candidate Pool Generation — v2_target_context

## What changed (v2)
- `TargetContext` builds ALL target-side structures **once per country** before the shard loop
- Rare-token inverted index: built 1× (was built 44× for US)
- Numeric inverted index: built 1× (was built 44× for US)
- Exact unique-name groupby: computed 1× (was computed 44× for US)
- GPU target matrix: uploaded 1× (was uploaded per shard)
- `ARCHITECTURE_FINGERPRINT` unchanged — candidates are semantically identical

## Gate: DO NOT RUN FULL until 2-shard steady-state test passes

## 01 — GPU environment

In [ ]:
import sys, os, subprocess
print(f"Python Version: {sys.version}")
print(f"CWD: {os.getcwd()}")
import subprocess
subprocess.run(['nvidia-smi'])
try:
    import cuml, cupy as cp
    print(f"cuML/CuPy is available. GPU_BACKEND_ACTIVE = TRUE")
    print(f"GPU free: {cp.cuda.Device().mem_info[0]/1e9:.2f} GB")
except ImportError:
    print("WARNING: cuML/CuPy not found — will fail at TF-IDF step.")


## 02 — Clone/reuse repo

In [ ]:
REPO_URL = "https://github.com/yugtheguy/amazon_ml.git"
REPO_DIR = "/kaggle/working/amazon_ml"
if not os.path.exists(REPO_DIR):
    print(f"Cloning {REPO_URL} ...")
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
else:
    print("Repo exists — pulling latest...")
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
os.chdir(REPO_DIR)
print(f"CWD: {os.getcwd()}")


In [ ]:
import subprocess
subprocess.run(["git", "status", "--short"])
r = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True)
print("Commit:", r.stdout.decode().strip())


## 03 — Processed data

In [ ]:
PROCESSED_DIR = None
known_path = "/kaggle/input/datasets/yugdeshmukh/amazon-ml-processed-v001"
if os.path.exists(os.path.join(known_path, "train_source1.parquet")):
    PROCESSED_DIR = known_path
    print(f"Processed data attached at: {PROCESSED_DIR}")
elif os.path.exists("data/processed/v001/train_source1.parquet"):
    PROCESSED_DIR = "data/processed/v001"
    print(f"Processed data found locally at: {PROCESSED_DIR}")
else:
    for root, dirs, files in os.walk("/kaggle/input"):
        if "train_source1.parquet" in files:
            PROCESSED_DIR = root
            break
    if PROCESSED_DIR:
        print(f"Found processed data at: {PROCESSED_DIR}")
    else:
        raise RuntimeError("No processed data found. Attach amazon-ml-processed-v001 dataset.")
print(f"PROCESSED_DIR = {PROCESSED_DIR}")


## 04 — Install deps + run tests

In [ ]:
import subprocess
subprocess.run(["pip", "install", "-r", "requirements.txt", "-q"])
r = subprocess.run(
    ["python", "-m", "pytest",
     "tests/test_c001_target_context.py",
     "tests/test_c001_invariants.py",
     "-v", "--tb=short"],
    env={**os.environ, "ALLOW_CPU_TFIDF": "1"},
    capture_output=True, text=True
)
print(r.stdout[-3000:] if len(r.stdout) > 3000 else r.stdout)
if r.returncode != 0:
    raise RuntimeError("TESTS FAILED — do not proceed")
print("ALL TESTS PASSED")


## 05 — Explicitly Disable CPU Fallback

Ensure production runs on GPU and fails fast if GPU is unavailable.

In [ ]:
import os
os.environ.pop("ALLOW_CPU_TFIDF", None)
assert os.environ.get("ALLOW_CPU_TFIDF") != "1", "CRITICAL: CPU fallback still enabled in os.environ!"
print("PRODUCTION_CPU_FALLBACK_ALLOWED = NO")


## 06 — C001_SMOKE_V2: Kaggle E2E Memory & Reuse Gate

Runs 10,000 US queries (two 5K shards) against the **FULL** US target datasets.
Verifies that `TargetContext` fits in VRAM and is properly reused for Shard B.

**Do not proceed to full run if this fails.**

In [ ]:
import subprocess, os
r = subprocess.run(
    ["python", "-u", "scripts/run_c001_smoke.py", "--processed-dir", PROCESSED_DIR],
    env={**os.environ, "PYTHONPATH": ".", "ALLOW_CPU_TFIDF": "0"},
    capture_output=False,
)
print(f"Exit code: {r.returncode}")
if r.returncode != 0:
    raise RuntimeError("SMOKE TEST FAILED — do not proceed")


## 07 — FULL RUN

**Only run this cell after the 2-shard gate passes.**

Runs all countries sequentially. Per-shard `.done` markers enable safe resume.
Estimated runtime with v2_target_context: **~2.5–4 hours** (down from ~8+ hours).

In [ ]:
import subprocess, os
OUT_DIR = "/kaggle/working/artifacts/candidate_pool"
DATA_DIR = "data"
r = subprocess.run(
    [
        "python", "-u", "scripts/run_c001_full_train.py",
        "--config", "configs/c001_full_train.yaml",
        "--data-dir", DATA_DIR,
        "--processed-dir", PROCESSED_DIR,
        "--out-dir", OUT_DIR,
        "--smoke-size", "0",
        "--chunk-size", "50000",
    ],
    env={**os.environ, "PYTHONPATH": ".", "ALLOW_CPU_TFIDF": "0"},
    capture_output=False,
)
print(f"Orchestrator exit code: {r.returncode}")
if r.returncode != 0:
    raise RuntimeError("FULL RUN FAILED")


## 07 — Evaluation metrics

In [ ]:
import json, os
metrics_path = f"{OUT_DIR}/C001/candidate_pool_v1/metrics/evaluation_metrics.json"
if os.path.exists(metrics_path):
    print(json.dumps(json.load(open(metrics_path)), indent=2))
else:
    print("Metrics not yet available — run may still be in progress.")


## 08 — Run manifest

In [ ]:
manifest_path = f"{OUT_DIR}/C001/candidate_pool_v1/manifest/run_manifest.json"
if os.path.exists(manifest_path):
    print(json.dumps(json.load(open(manifest_path)), indent=2))


## 09 — Package artifacts

In [ ]:
import shutil, os
bundle_path = f"{OUT_DIR}/C001/candidate_pool_v1/candidate_pool_v1_bundle.tar.gz"
if os.path.exists(bundle_path):
    size_mb = os.path.getsize(bundle_path) / 1e6
    print(f"Bundle ready: {bundle_path} ({size_mb:.1f} MB)")
else:
    print("Bundle not yet created.")
